# Setup And Index


In [1]:
%%capture
!pip install -q -r requirements.txt
!rm -rf repo

In [2]:
import os, subprocess, sys

if not os.path.exists("portfolio.py"):
    if os.path.exists("../portfolio.py"):
        os.chdir("..")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/hossamhamdy333/AI_Portfolio.git", "repo"],
            check=True,
        )
        os.chdir("repo/Codebase_Insight_Agent")

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())


Working directory: /content/repo/Codebase_Insight_Agent


In [7]:
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google API key (Gemini): ")
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = input("Qdrant Cloud URL: ")
if os.environ["QDRANT_URL"] and not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")


In [4]:
!pip install qdrant-client
!pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai
!pip install llama-index-vector-stores-qdrant
!pip install llama-index-embeddings-huggingface

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import json
import time
import portfolio
from pathlib import Path
import config

CHECKPOINT = Path('/content/drive/MyDrive/indexing_progress.json')
done = set(json.loads(CHECKPOINT.read_text())) if CHECKPOINT.exists() else set()

client = portfolio.get_qdrant_client()
indexes = {}

for name in config.PROJECTS:
    if name in done:
        print(f"{name}: already rebuilt this pass, skipping")
        indexes[name] = portfolio.load_index(name, client)
        continue

    print(f"Rebuilding {name}...")
    indexes[name] = portfolio.build_index(name, client)
    done.add(name)
    CHECKPOINT.write_text(json.dumps(sorted(done)))
    time.sleep(5)

print(f"Done. {len(done)}/{len(config.PROJECTS)} rebuilt this pass.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rebuilding customer_support_copilot...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Rebuilding Azure_RAG_Assistant...
Rebuilding Credit_Fraud_Detection...
Rebuilding House_Price_Prediction...
Rebuilding allam_finetune...
Rebuilding customer_churn_prediction...
Rebuilding fact_check_crew...
Rebuilding rag_router...
Rebuilding rag-vanilla-vs-langchain...
Rebuilding nl2sql_finetune...
Rebuilding ecommerce-demand-forecasting...
Rebuilding employee-attrition...
Rebuilding llm_api_integration...
Rebuilding marketing-ab-testing...
Rebuilding rag_qa_documind...
Rebuilding semantic-search-arxiv-papers...
Rebuilding sentiment_forge...
Rebuilding machine-learning-techniques-for-intrusion-detection...
Rebuilding ids-deploy...
Done. 19/19 rebuilt this pass.


Quick sanity check

In [10]:
import nest_asyncio
nest_asyncio.apply()

engine = indexes["Credit_Fraud_Detection"].as_query_engine()

response = engine.query("What is this project about?")
print(response)

This project is focused on the detection of credit fraud.
